# megatron中是如何使用torch.nn.parallel（DDP）

- 在 Megatron-LM 中，torch.nn.parallel.DistributedDataParallel（DDP）的使用方式与标准 PyTorch 有所不同。
- Megatron 采用了自定义的分布式策略，将 DDP 与张量并行（TP）、流水线并行（PP）深度集成。

## Megatron 的并行架构

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    Megatron-LM 3D 并行                         │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   数据并行 (DP)  ──→  torch.nn.parallel.DistributedDataParallel │
│   张量并行 (TP)  ──→  megatron.core.tensor_parallel          │
│   流水线并行 (PP) ──→  megatron.core.pipeline_parallel        │
│                                                              │
│   组合方式：DP × TP × PP = 总 GPU 数                         │
│                                                              │
│   示例：8 节点 × 8 GPU = 64 GPU                             │
│          DP=4, TP=8, PP=2  →  4 × 8 × 2 = 64               │
└─────────────────────────────────────────────────────────────┘

## Megatron 中 DDP 的使用位置
### 标准用法：在 Pipeline Stage 内部
- 在 Megatron 中，DDP 通常不直接包裹整个模型，而是包裹每个流水线阶段（pipeline stage）内的模型块：

In [ ]:
# Megatron-LM 简化的模型初始化流程

from torch.nn.parallel import DistributedDataParallel as DDP
from megatron.core import parallel_state

def get_model(model_provider_func):
    """
    创建并并行化模型
    """
    # 1. 获取流水线并行配置
    pp_size = parallel_state.get_pipeline_model_parallel_world_size()
    pp_rank = parallel_state.get_pipeline_model_parallel_rank()
    
    # 2. 创建当前 pipeline stage 的模型
    model = model_provider_func()  # 用户提供的模型构建函数
    
    # 3. 应用张量并行（TP）
    #    在 TP group 内拆分线性层
    model = tensor_parallelize_model(model)
    
    # 4. 应用 DDP（数据并行）
    #    只在非张量并行、非流水线并行的维度上使用 DDP
    if parallel_state.get_data_parallel_world_size() > 1:
        model = DDP(
            model,
            process_group=parallel_state.get_data_parallel_group(),
            # 关键：DDP 只在 DP group 内同步梯度
            # TP 和 PP 的梯度由各自的机制处理
        )
    
    return model

## Megatron 的自定义 DDP 实现
### 不使用标准 DDP，而是自定义梯度同步
<font color='red'>Megatron-LM 实际上很少直接使用 torch.nn.parallel.DDP，而是使用自定义的分布式数据并行（DistributedDataParallel 的简化版或完全自定义实现），原因：</font>
- TP 已经同步了部分梯度（All-Reduce in TP group）
- PP 的梯度通过 P2P 通信传递
- 需要精细控制梯度 bucket 和通信时机

In [ ]:
# Megatron 核心：自定义梯度同步（简化示意）

from megatron.core.distributed import DistributedDataParallel as MegatronDDP

class DistributedDataParallel(MegatronDDP):
    """
    Megatron 的自定义 DDP
    特点：
    1. 只在数据并行组内同步
    2. 与张量并行、流水线并行协调
    3. 支持梯度累积、梯度裁剪
    4. 支持 FP16/BF16 混合精度
    """
    
    def __init__(self, module, config, **kwargs):
        super().__init__()
        self.module = module
        
        # 获取数据并行组
        self.data_parallel_group = parallel_state.get_data_parallel_group()
        self.data_parallel_rank = parallel_state.get_data_parallel_rank()
        self.data_parallel_world_size = parallel_state.get_data_parallel_world_size()
        
        # 注册梯度钩子
        for param in module.parameters():
            if param.requires_grad:
                param.register_hook(self._grad_hook)
    
    def _grad_hook(self, grad):
        """
        梯度计算完成后自动触发：
        在 DP group 内 All-Reduce 梯度
        """
        if self.data_parallel_world_size > 1:
            # 只在数据并行组内同步
            # 注意：TP 和 PP 的梯度不在这里同步
            torch.distributed.all_reduce(
                grad,
                group=self.data_parallel_group,
                async_op=True  # 异步，重叠计算
            )
        return grad
    
    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)

## 实际代码：Megatron 中的模型并行化
### 核心函数：get_model 和 model_provider_func

In [ ]:
# Megatron-LM/megatron/training.py（简化）

from megatron.core.distributed import DistributedDataParallel
from megatron.core.transformer.module import Float16Module

def get_model(model_provider_func, model_type=ModelType.encoder_or_decoder):
    """
    构建并并行化模型
    """
    args = get_args()
    
    # 1. 创建模型（用户自定义的 model_provider）
    model = model_provider_func(
        pre_process=parallel_state.is_pipeline_first_stage(),
        post_process=parallel_state.is_pipeline_last_stage(),
    )
    
    # 2. 应用张量并行（TP）
    #    自动拆分 attention、MLP 中的线性层
    #    使用 ColumnParallelLinear / RowParallelLinear
    
    # 3. 应用流水线并行（PP）
    #    将模型切分为多个 stage
    #    当前进程只持有自己 stage 的层
    
    # 4. 应用数据并行（DP）—— 这里用到 DDP 思想
    if not args.distribute_saved_activations:
        # 标准 DDP 包装
        model = DistributedDataParallel(
            model,
            config=args,
            accumulate_allreduce_grads_in_fp32=args.accumulate_allreduce_grads_in_fp32,
            overlap_grad_reduce=args.overlap_grad_reduce,
            use_contiguous_buffers_in_local_ddp=args.use_contiguous_buffers_in_local_ddp,
        )
    
    # 5. 可选：FP16/BF16 包装
    if args.fp16 or args.bf16:
        model = Float16Module(model, args)
    
    return model

## 关键：Megatron 的梯度同步机制
### 三种并行的梯度处理

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    梯度同步策略                                │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│  张量并行 (TP)                                               │
│  ┌─────────────────────────────────────────────────────┐    │
│  │ ColumnParallelLinear / RowParallelLinear           │    │
│  │ 前向：输入在 TP group 内广播                          │    │
│  │ 反向：梯度在 TP group 内 All-Reduce                  │    │
│  │ 实现：torch.distributed.all_reduce(group=tp_group)   │    │
│  └─────────────────────────────────────────────────────┘    │
│                                                              │
│  流水线并行 (PP)                                             │
│  ┌─────────────────────────────────────────────────────┐    │
│  │ 1F1B 调度                                            │    │
│  │ 前向：P2P 发送激活值到下一 stage                      │    │
│  │ 反向：P2P 发送梯度到上一 stage                        │    │
│  │ 实现：torch.distributed.send/recv                     │    │
│  └─────────────────────────────────────────────────────┘    │
│                                                              │
│  数据并行 (DP)                                               │
│  ┌─────────────────────────────────────────────────────┐    │
│  │ DDP 或自定义梯度同步                                  │    │
│  │ 反向完成后：在 DP group 内 All-Reduce 梯度            │    │
│  │ 实现：Megatron 自定义（非标准 torch DDP）             │    │
│  └─────────────────────────────────────────────────────┘    │
│                                                              │
└─────────────────────────────────────────────────────────────┘

### 梯度同步代码（Megatron 实际实现）

In [ ]:
# Megatron 核心：梯度 reduce-scatter / all-reduce

def allreduce_gradients(self):
    """
    在数据并行组内同步梯度
    被 optimizer.step() 前调用
    """
    # 1. 收集需要同步的梯度
    grads = []
    for param in self.module.parameters():
        if param.grad is not None:
            grads.append(param.grad)
    
    # 2. 按 bucket 分组（类似 DDP 的 gradient bucketing）
    buckets = _create_buckets(grads, bucket_size_mb=25)
    
    # 3. 异步 All-Reduce 每个 bucket
    handles = []
    for bucket in buckets:
        # 重叠：一个 bucket 通信时，下一个 bucket 准备
        handle = torch.distributed.all_reduce(
            bucket,
            group=self.data_parallel_group,
            async_op=True
        )
        handles.append(handle)
    
    # 4. 等待所有通信完成
    for handle in handles:
        handle.wait()
    
    # 5. 平均梯度
    for grad in grads:
        grad /= self.data_parallel_world_size

## Megatron 与标准 DDP 的对比

| 特性            | 标准 PyTorch DDP | Megatron 自定义实现         |
| ------------- | -------------- | ---------------------- |
| **包裹范围**      | 整个模型           | 每个 Pipeline Stage      |
| **梯度同步范围**    | 所有参数           | 只在 DP group 内          |
| **TP 梯度处理**   | 不支持            | 内置 Column/Row Parallel |
| **PP 梯度处理**   | 不支持            | 内置 P2P 通信              |
| **通信重叠**      | 自动 bucket      | 自定义，与 PP 调度配合          |
| **FP16/BF16** | 需额外配置          | 内置 Float16Module       |
| **梯度累积**      | 需手动实现          | 内置支持                   |


## 实际使用示例
### 用户侧代码（使用 Megatron 训练 GPT）

In [ ]:
# 用户只需提供 model_provider，Megatron 内部处理并行

from megatron.training import get_model, pretrain
from megatron.core.transformer import GPTModel

def gpt_model_provider(pre_process=True, post_process=True):
    """
    用户定义：如何构建模型
    Megatron 会自动应用 TP/PP/DP
    """
    config = TransformerConfig(
        num_layers=96,
        hidden_size=12288,
        num_attention_heads=96,
        # ...
    )
    
    model = GPTModel(
        config=config,
        pre_process=pre_process,
        post_process=post_process,
    )
    
    return model

# Megatron 内部会自动：
# 1. 应用 TP（拆分 attention/MLP）
# 2. 应用 PP（切分 layer 到不同 stage）
# 3. 应用 DP（自定义梯度同步）
# 用户无需手动调用 torch.nn.parallel.DDP

if __name__ == '__main__':
    pretrain(
        train_valid_test_dataset_provider,
        gpt_model_provider,
        ModelType.encoder_or_decoder,
        forward_step,
        args_defaults={'tokenizer_type': 'GPT2BPETokenizer'}
    )

### 底层：Megatron 的 DDP 类定义

In [ ]:
# megatron/core/distributed/distributed_data_parallel.py

class DistributedDataParallel(MegatronModule):
    """
    Megatron 的 DDP 实现
    不是继承 torch.nn.parallel.DistributedDataParallel
    而是自己实现梯度同步逻辑
    """
    
    def __init__(
        self,
        module,
        config,
        accumulate_allreduce_grads_in_fp32=True,
        overlap_grad_reduce=True,
        use_contiguous_buffers_in_local_ddp=True,
    ):
        super().__init__()
        self.module = module
        self.config = config
        
        # 数据并行组
        self.data_parallel_group = parallel_state.get_data_parallel_group()
        self.data_parallel_world_size = parallel_state.get_data_parallel_world_size()
        self.data_parallel_rank = parallel_state.get_data_parallel_rank()
        
        # 重叠通信和计算
        self.overlap_grad_reduce = overlap_grad_reduce
        
        # 注册梯度钩子
        self._register_grad_hooks()
        
        # 创建连续缓冲区（优化内存）
        if use_contiguous_buffers_in_local_ddp:
            self._create_contiguous_buffers()
    
    def _register_grad_hooks(self):
        """注册反向传播钩子，用于梯度同步"""
        for param in self.module.parameters():
            if param.requires_grad:
                # 梯度计算完成后触发 all-reduce
                param.register_post_accumulate_grad_hook(
                    self._make_param_hook(param)
                )
    
    def _make_param_hook(self, param):
        def hook(*unused):
            if self.overlap_grad_reduce:
                # 异步 all-reduce，不阻塞计算
                param.grad_handle = torch.distributed.all_reduce(
                    param.grad,
                    group=self.data_parallel_group,
                    async_op=True
                )
            else:
                # 同步 all-reduce
                torch.distributed.all_reduce(
                    param.grad,
                    group=self.data_parallel_group
                )
        return hook
    
    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)
    
    def finish_grad_sync(self):
        """等待所有异步梯度同步完成"""
        for param in self.module.parameters():
            if hasattr(param, 'grad_handle') and param.grad_handle is not None:
                param.grad_handle.wait()
                param.grad_handle = None

## 总结

| 问题                                    | 答案                                                       |
| ------------------------------------- | -------------------------------------------------------- |
| Megatron 用 `torch.nn.parallel.DDP` 吗？ | **基本不用**，使用自定义实现                                         |
| 为什么不用标准 DDP？                          | 需要与 TP/PP 深度集成，标准 DDP 无法满足                               |
| Megatron 的 DDP 在哪里？                   | `megatron/core/distributed/distributed_data_parallel.py` |
| 用户需要手动调用 DDP 吗？                       | **不需要**，Megatron 内部自动处理                                  |
| 梯度同步机制                                | 自定义钩子 + 异步 All-Reduce + Bucket 优化                        |
- 核心设计哲学：
  - Megatron 将 torch.nn.parallel.DistributedDataParallel 的思想（数据并行、梯度同步）内化到自己的架构中，但具体实现完全自定义，以支持 3D 并行的复杂需求。